# OGBN-ArXiv Feature Engineering for Traditional ML

This notebook performs feature engineering on the OGBN-ArXiv dataset to preserve graph structural information in tabular format. The goal is to extract graph topology features that can be used with traditional ML algorithms like XGBoost and standard neural networks, avoiding the need for specialized graph neural networks.

In [1]:
from ogb.nodeproppred import NodePropPredDataset
import pandas as pd
import networkx as nx

## Load the OGBN-ArXiv Dataset
The dataset contains:
- **Nodes**: ~169K research papers
- **Edges**: ~1.2M citation links
- **Node Features**: 128-dimensional embeddings derived from paper abstracts
- **Labels**: 40 subject area classes

In [2]:
dataset = NodePropPredDataset(name="ogbn-arxiv", root="../../data/ogbn")
graph, labels = dataset[0]

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:06<00:00, 12.50it/s]


Extracting ../../data/ogbn\arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<?, ?it/s]

Saving...


## Extract Graph Topology as Tabular Features

We convert the edge list to a NetworkX graph and compute structural features that capture the graph topology in tabular format. This allows us to preserve critical graph information for use with traditional ML algorithms like XGBoost and standard neural networks.

### Graph Features Extracted:
1. **Degree Centrality**: Number of connections (citations and cited by) for each paper
   - Captures local connectivity and paper influence
   - Essential for XGBoost to understand node importance

2. **PageRank**: Global importance based on the entire citation network
   - Provides a recursive measure of paper significance
   - Helps traditional ML models understand network-wide influence patterns

3. **Clustering Coefficient**: Local network density around each node
   - Captures community structure information
   - Indicates whether papers are in dense research clusters

4. **Betweenness Centrality**: Measures a bridging role between different parts of the network
   - Identifies papers that connect different research areas
   - Critical for understanding interdisciplinary connections

In [3]:
edge_index = graph["edge_index"]
node_feat = graph["node_feat"]

In [ ]:
G = nx.Graph()
G.add_edges_from(edge_index.T.tolist())

features = {
    'degree': dict(G.degree()),
    'pagerank': nx.pagerank(G),
    'clustering': nx.clustering(G),
    'betweenness': nx.betweenness_centrality(G),
}

df_graph = pd.DataFrame(features)
df_graph.index.name = 'node_id'

## Create ML-Ready Dataset

We combine the original node features with our extracted graph topology features to create a comprehensive tabular dataset:

1. **Content Features**: 128-dimensional embeddings from paper abstracts
2. **Graph Topology Features**: Structural properties that capture the citation network
3. **Labels**: Ground truth subject area classifications

In [ ]:
df_feats = pd.DataFrame(node_feat)
df_feats['label'] = labels.flatten()
df_feats['node_id'] = df_feats.index

data = df_feats.merge(df_graph, left_on='node_id')

## Inspect the Final Feature Set

Let's examine the structure of our final feature set to ensure everything looks correct.

In [ ]:
data.keys()

## Save the ML-Ready Dataset

In [ ]:
data.to_csv('../../data/obgn/processed/obgn_arxiv_processed.csv', index=False)